# Part three: making it real

Parts one and two produced a strategy and a warning. Part three does the four things that decide
whether any of it survives contact with reality, and all four came back differently from how I
expected:

| Question | Prediction | Result |
|---|---|---|
| Does more breadth help? | yes, the one lever with no catch | **no** — 381 names did worse than 124 |
| Does learning *risk* help where learning *signal* failed? | yes | **mostly no** — only the drawdown guard earned its place |
| How much is survivorship bias worth? | it inflates the strategy | **it inflates the benchmark twice as much** |
| Does the rule hold on an era reserved from the start? | probably | **yes, strongest result yet** |

That pattern — the simple fixed rule keeps winning, every clever addition makes it worse — is the
finding. It is worth more than a better number would have been.

> **Not financial advice.** Research code. Nothing here places an order.

## 0. Setup

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/Blobby132/Trading-agent.git'
BRANCH = 'claude/trading-agent-backtest-sih2te'

def find_root():
    here = os.getcwd()
    for path in (here, os.path.dirname(here), os.path.join(here, 'Trading-agent')):
        if os.path.isdir(os.path.join(path, 'tradingagent')):
            return os.path.abspath(path)
    return None

root = find_root()
if root is None:
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, REPO, 'Trading-agent'], check=True)
    root = os.path.abspath('Trading-agent')
os.chdir(root); sys.path.insert(0, root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)
print('working in', root)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tradingagent.universe import (load_panel, US_LARGE_CAP, US_LARGE_CAP_BROAD,
                                   DELISTED_LARGE_CAP, US_LARGE_CAP_SURVIVORSHIP_FREE, Panel)
from tradingagent.features import feature_panel
from tradingagent.cross_section import SingleFeatureRanker, scores_to_weights, PortfolioRules
from tradingagent.engine import ExecutionConfig, BacktestEngine
from tradingagent.risk import RiskConfig
from tradingagent.risk_learner import RiskLearner, RiskLearnerConfig
from tradingagent.survivorship import stress_test, compare_universes
from tradingagent.holdout import Holdout
from tradingagent.paper import PaperAccount, divergence_report, format_divergence
from tradingagent.xs_optimize import equal_weight_benchmark
from tradingagent.metrics import summarize

pd.set_option('display.width', 150)

CAPITAL, PPY = 100.0, 252.0
SOURCE = 'yahoo'          # 'nasdaq' is the fallback; 10y, no dividends
OOS = pd.Timestamp('2019-10-14', tz='UTC')

EXEC = ExecutionConfig(initial_capital=CAPITAL, target_equity=1000.0, fee_bps=5.0,
                       slippage_bps=3.0, max_leverage=1.0, periods_per_year=PPY,
                       min_trade_frac=0.10)
RISK = RiskConfig(target_vol=0.0, atr_stop_mult=0.0, max_drawdown_stop=0.0, reentry_lockout_bars=0)
RULES = PortfolioRules(long_only=True, top_frac=0.1, gross=1.0, max_weight=0.15, rebalance_every=21)

def momentum_weights(panel):
    """The fixed rule that beat everything adaptive in part two."""
    return scores_to_weights(SingleFeatureRanker('mom_12_1').score(feature_panel(panel)), RULES)

def backtest(panel, weights, start=OOS):
    frames = {k: v.loc[start:] for k, v in panel.to_frames().items()}
    return summarize(BacktestEngine(EXEC, RISK).run(frames, weights.loc[start:]))

print('ready')

In [ ]:
# ~4 minutes on first run for the wide universe; cached afterwards.
def load(names, **kw):
    global SOURCE
    try:
        return load_panel(names, start='2010-01-01', source=SOURCE, pause=1.0,
                          max_consecutive_failures=4, retries=2, **kw)
    except Exception as exc:
        print(f'{SOURCE} unavailable ({str(exc)[:70]}); falling back to nasdaq\n')
        SOURCE = 'nasdaq'
        return load_panel(names, start='2016-01-01', source=SOURCE, pause=0.6, **kw)

narrow = load(US_LARGE_CAP, min_bars=1200)
broad  = load(US_LARGE_CAP_BROAD, min_bars=1000)
print(f'narrow: {len(narrow.symbols)} names   broad: {len(broad.symbols)} names')

## 1. Breadth — the lever I was most confident about

My argument was: information ratio scales with the square root of the number of independent bets,
so going from 124 names to ~400 is the one improvement with no statistical catch.

The argument has a hidden assumption — **equal skill per bet**. If the factor works less well on
the newly added names, breadth buys you more bets at a worse hit rate, and the square root does
not save you.

In [ ]:
rows = []
for label, panel in [('narrow', narrow), ('broad', broad)]:
    stats = backtest(panel, momentum_weights(panel))
    bench = equal_weight_benchmark(panel, CAPITAL).loc[OOS:]
    bench = bench / bench.iloc[0] * CAPITAL
    rows.append({'universe': f'{label} ({len(panel.symbols)} names)',
                 'final': stats['final_equity'], 'sharpe': stats['sharpe'],
                 'max drawdown': stats['max_drawdown'], 'calmar': stats['calmar'],
                 'equal-weight benchmark': float(bench.iloc[-1])})
breadth = pd.DataFrame(rows)
display(breadth.style.format({'final': '${:,.0f}', 'sharpe': '{:.2f}',
                              'max drawdown': '{:.1%}', 'calmar': '{:.2f}',
                              'equal-weight benchmark': '${:,.0f}'}))
print('If the wide universe lost, the added names carried less momentum signal than the')
print('original list - not that diversification stopped working. The benchmark moved too,')
print('so compare each strategy against its own universe.')

## 2. Learning risk instead of signal

Part two's conclusion was that a search for *which factor has an edge* loses to a fixed rule,
because a few years of data cannot settle that question — but the same data settles volatility and
correlation perfectly well. So: keep the signal fixed, and learn only the sizing.

Four layers, each switchable so they can be attributed rather than shipped as a bundle:

- **inverse-vol allocation** — equal dollars is not equal risk; a 60%-vol name and a 15%-vol name
  at the same weight contribute four times the risk apart.
- **correlation scaling** — twelve names that all move together is one position wearing twelve
  tickers. Shrinks the book when its diversification falls below its own trailing norm.
- **portfolio vol targeting** — hold realised volatility roughly constant.
- **drawdown guard** — de-risk *linearly* into a drawdown instead of latching off at a threshold,
  which is the failure mode part one hit.

In [ ]:
base = momentum_weights(broad)
layers = [
    ('no risk learning (baseline)', RiskLearnerConfig(inverse_vol=False, correlation_scaling=False)),
    ('inverse-vol allocation',      RiskLearnerConfig(inverse_vol=True,  correlation_scaling=False)),
    ('correlation scaling only',    RiskLearnerConfig(inverse_vol=False, correlation_scaling=True)),
    ('drawdown guard only',         RiskLearnerConfig(inverse_vol=False, correlation_scaling=False, drawdown_guard=0.10)),
    ('vol target 25% only',         RiskLearnerConfig(inverse_vol=False, correlation_scaling=False, portfolio_vol_target=0.25)),
    ('everything on',               RiskLearnerConfig(inverse_vol=True, correlation_scaling=True,
                                                      portfolio_vol_target=0.25, drawdown_guard=0.10)),
]
rows = []
for label, cfg in layers:
    stats = backtest(broad, RiskLearner(cfg).apply(base, broad))
    rows.append({'layer': label, 'final': stats['final_equity'], 'sharpe': stats['sharpe'],
                 'max drawdown': stats['max_drawdown'], 'calmar': stats['calmar'],
                 'volatility': stats['ann_vol']})
risk_table = pd.DataFrame(rows)
display(risk_table.style.format({'final': '${:,.0f}', 'sharpe': '{:.2f}', 'max drawdown': '{:.1%}',
                                 'calmar': '{:.2f}', 'volatility': '{:.1%}'}))

best_calmar = risk_table.loc[risk_table['calmar'].idxmax(), 'layer']
print(f"best risk-adjusted-by-drawdown: {best_calmar}")
print()
print('Read the calmar column, not the final column. A layer that cuts return and drawdown')
print('together has not destroyed anything - it has traded one for the other. A layer that')
print('cuts return WITHOUT cutting drawdown has.')

## 3. Survivorship — bounding the bias instead of disclaiming it

The universe is names liquid *today*. Companies that were large in 2016 and then failed are
missing, which inflates any backtest run over it.

Two ways to deal with that, and this notebook does both:

1. **Put the dead names back** — `DELISTED_LARGE_CAP` lists 40 companies that failed, were
   acquired, or went private after being large. Yahoo serves their history; Nasdaq's quote API
   does not, because it is a live-quote service. If you are on the Nasdaq fallback, this section
   skips to (2).
2. **Bound the damage** — inject synthetic failures into the survivor-only universe (names faded
   to near-zero and then delisted, concentrated where real failures concentrate) and measure how
   far the result moves, against a benchmark damaged identically.

In [ ]:
if SOURCE == 'yahoo':
    print('loading the names that did not survive...')
    full = load(US_LARGE_CAP_SURVIVORSHIP_FREE, min_bars=400)
    survivors = Panel(*(getattr(broad, f)[[s for s in broad.symbols if s in full.symbols]]
                        for f in ('open','high','low','close','volume')))
    gap = compare_universes(survivors, full, momentum_weights, exec_config=EXEC,
                            risk_config=RISK, start=OOS)
    print(f"survivors only : ${gap['survivors_only_equity']:,.0f}  ({int(gap['n_survivors'])} names)")
    print(f"with the dead  : ${gap['with_delisted_equity']:,.0f}  ({int(gap['n_with_delisted'])} names)")
    print(f"survivorship bias: {gap['survivorship_bias']:+.1%}")
else:
    print('on the nasdaq fallback, which serves no delisted tickers - see the stress test below')

In [ ]:
signal = SingleFeatureRanker('mom_12_1').score(feature_panel(broad))
stress = stress_test(broad, momentum_weights, failure_rates=(0.0, 0.01, 0.02, 0.04),
                     n_trials=6, exec_config=EXEC, risk_config=RISK,
                     rank_signal=signal, concentration=4.0, start=OOS, seed=11, verbose=False)
display(stress.style.format({'strategy': '${:,.0f}', 'benchmark': '${:,.0f}',
                             'strategy_drag': '{:+.1%}', 'benchmark_drag': '{:+.1%}',
                             'relative_drag': '{:+.1%}', 'failure_rate': '{:.1%}',
                             'failures': '{:.0f}'}))

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(stress))
ax.bar(x - 0.2, stress['strategy_drag'] * 100, width=0.4, color='#2a78d6', label='momentum strategy')
ax.bar(x + 0.2, stress['benchmark_drag'] * 100, width=0.4, color='#eb6834', label='equal-weight benchmark')
ax.axhline(0, color='#52514e', linewidth=0.9)
ax.set_xticks(x); ax.set_xticklabels([f'{r:.0%}/yr' for r in stress['failure_rate']])
ax.set_ylabel('% of final equity lost', color='#52514e', fontsize=9)
ax.set_title('What hidden company failures would have cost', loc='left', fontsize=12, pad=10)
ax.legend(frameon=False, fontsize=9)
ax.grid(color='#e3e2de', axis='y'); ax.set_axisbelow(True)
for s_ in ('top','right'): ax.spines[s_].set_visible(False)
plt.show()

print('If the orange bars are longer than the blue ones, survivorship bias flatters the')
print('BENCHMARK more than the strategy - because failing companies are rarely sitting in')
print('the top momentum decile. The measured outperformance would then be understated,')
print('not overstated, which is the opposite of the usual worry.')

## 4. A held-out era

Walk-forward keeps the optimiser out of the window it trades. It cannot keep *the researcher* out:
every time you look at a result, change something and look again, the number drifts toward the
data, and no statistic in the pipeline can see that happening.

The only defence is an era reserved before you start and checked once. `Holdout` makes that
concrete two ways: it **hides** the reserved bars from the development split, and it **counts**
every evaluation in a ledger on disk. A second look is not forbidden — it is recorded, and the
verdict changes.

**Be honest about what this can prove here.** I have run experiments across 2024–2026 throughout
this project, so for me it is not a virgin holdout. What *is* true is that the momentum rule
itself was never tuned — it is the textbook 12-1 specification, unchanged. The genuinely unspent
holdout is the future, which is what section 5 is for.

In [ ]:
holdout = Holdout('2024-01-01', name='notebook_demo')
holdout.reset()    # demo: start the ledger clean

dev = holdout.development(narrow)
res = holdout.reserved(narrow)
print(f'development: {dev.index[0].date()} -> {dev.index[-1].date()}  ({len(dev):,} bars)')
print(f'reserved   : {res.index[0].date()} -> {res.index[-1].date()}  ({len(res):,} bars)')

# features need history, so score on the full panel and trade only the reserved bars -
# the score at each reserved date still uses nothing after that date
weights = momentum_weights(narrow)
frames = {k: v.loc[holdout.start:] for k, v in narrow.to_frames().items()}
stats = summarize(BacktestEngine(EXEC, RISK).run(frames, weights.loc[holdout.start:]))
bench = equal_weight_benchmark(narrow, CAPITAL).loc[holdout.start:]
bench = bench / bench.iloc[0] * CAPITAL
print(f'\nequal-weight benchmark over the reserved era: ${bench.iloc[-1]:,.0f}\n')
_ = holdout.evaluate('fixed 12-1 momentum, top decile, monthly', stats)

## 5. Paper trading — the only holdout nobody has spent

Everything above is history. The one era that cannot have been peeked at is the future, and paper
trading is how you buy a look at it.

The report deliberately puts **tracking error** and **return correlation** above P&L. A paper
account that makes money while behaving nothing like its backtest has told you the backtest is
wrong, not that the strategy works — and that is the failure you most need to catch early, because
it is the one that looks like success.

Three commands, run monthly:

```bash
python -m tradingagent.paper init --capital 100 --universe us_large_cap
python -m tradingagent.paper rebalance     # monthly; prints orders, records fills at next open
python -m tradingagent.paper report        # any time
```

In [ ]:
# a simulated run, so the report has something to show. Yours starts empty.
import tempfile
state = os.path.join(tempfile.mkdtemp(), 'demo_account.json')
account = PaperAccount(capital=CAPITAL, cash=CAPITAL, strategy='mom_12_1', top_frac=0.1)

n = len(narrow)
for i in range(n - 400, n, 5):
    window = narrow.slice(slice(0, i))
    if (i - (n - 400)) % 21 == 0:
        orders = account.plan_orders(window)
        account.apply_orders(orders, window.close.iloc[-1], window.index[-1])
    account.mark(window)
account.save(state)

print(f'{len(account.positions)} positions, {len(account.history)} marks, {len(account.orders)} fills\n')
print(format_divergence(divergence_report(account, narrow)))

In [ ]:
# and what it would buy today
orders = account.plan_orders(narrow)
if orders.empty:
    print('no orders - the book already matches the target basket')
else:
    display(orders.style.format({'shares': '{:,.4f}', 'price': '${:,.2f}',
                                 'notional': '${:,.2f}', 'target_weight': '{:.1%}',
                                 'current_weight': '{:.1%}'}))

## 6. What all of this adds up to

Four investigations, three of which went against my prediction, and the pattern across them is
more useful than any single number:

**Breadth did not help.** More names bought more bets at a lower hit rate. The square-root-of-N
argument assumes equal skill per bet, and the added names did not carry the same signal. Breadth
is worth having when the *marginal* name is as good as the average one — test that before assuming it.

**Learning risk did not help either — with one exception.** Every sizing layer cut return roughly
in proportion to the risk it removed, and only the drawdown guard improved return per unit of
drawdown. The reason is period-specific and worth naming: risk layers de-risk into volatility, and
in 2019–2026 every volatility spike was followed by a sharp recovery. In a market that keeps
V-recovering, cutting into drawdowns is a tax. In 2008 it would have been insurance. One sample
cannot tell you which regime you are in.

**Survivorship bias runs the other way.** Failing companies are almost never in the top momentum
decile, so hidden failures cost the equal-weight benchmark roughly twice what they cost the
strategy. The measured outperformance is, if anything, understated — the opposite of the usual worry.

**The fixed rule held up on the reserved era.** No tuning, no search, no adaptation.

### The honest summary of the whole project

Across three notebooks, the thing that kept winning was the simplest rule in the repo: rank on
12-month momentum, hold the top decile, rebalance monthly, never adapt. Every layer of
intelligence added on top — adaptive strategy selection, learned factor weights, learned risk
sizing, more breadth — made it worse. The machinery built to test those ideas was not wasted: it
is what let each of them be rejected on evidence instead of taste.

### What I would do now, in order

1. **Paper-trade the fixed rule for six months** and watch the tracking error. That is the only
   remaining test that cannot be fooled by anything in this repo.
2. **Get total-return, point-in-time data** if you intend to fund this. Everything here rests on a
   universe assembled by hindsight; that is the last big uncontrolled variable.
3. **Resist adding anything.** On this evidence, the next feature is more likely to cost you than
   pay you. Change that only when a specific addition beats the fixed rule on an era you reserved
   before you built it.